# Calibration Dataset Construction

This notebook constructs a small calibration dataset consisting of ten OCT
B-scans.

Five scans are selected from fast progressors and five from slow
progressors.

For every scan the notebook performs:

1. Heidelberg E2E loading.
2. Anatomical preprocessing.
3. Structural-HyperTD detection.
4. Visualization of the preprocessed scan.
5. Visualization of the detected barcode intervals.

These images will subsequently be manually annotated and used to tune the
detector parameters.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from src.detector.data import build_grouped_volume_registry, load_e2e_volume
from src.detector.preprocessing import preprocess_bscan
from src.detector.detector import detect_structural_hypertransmission

print( "Project root:",PROJECT_ROOT)

In [ ]:
E2E_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "heyex"
    / "meta"
)

registry = build_grouped_volume_registry(
    e2e_directory=E2E_DIRECTORY,
    progression_groups={
        "fast": [8, 9, 12, 41, 49],
        "slow": [17, 23, 35, 36, 47],
    },
)

In [ ]:
SCAN_CONFIG = {
    "fast": {
        8: 48,
        9: 85,
        12: 38,
        41: 59,
        49: 33,
    },

    "slow": {
        17: 55,
        23: 43,
        35: 38,
        36: 59,
        47: 31,
    },
}

In [ ]:
PREPROCESSING_CONFIG = {
    # Retinal layer used for anatomical alignment.
    "layer_name": "BM",

    # Flattening
    # Target row for the selected retinal layer after flattening.
    #   None: automatically use the median layer position.
    #   Larger values move the flattened layer lower in the image.
    "reference_row": None,

    # Pixel value assigned to regions created after vertical shifting.
    #   Usually left at zero.
    "flatten_fill_value": 0.0,


    # Sub-layer crop
    # Number of pixels retained beneath the selected retinal layer.
    #   +: Includes more deep tissue and choroid.
    #   -: Restricts analysis to tissue immediately below the layer.
    "depth_below_layer": 150,

    # Whether the reference layer itself is included in the crop.
    #   False begins one pixel below the layer.
    "include_boundary": True,

    # T: scans without sufficient depth are rejected.
    # F: shallow scans are padded.
    "require_full_depth": False,

    # Pixel value used when crop padding is required.
    "crop_fill_value": 0.0,


    # Intensity normalization method.
    #   "zscore"
    #   "minmax"
    #   "percentile"
    "normalization_method": "zscore",

    # Lower percentile used by percentile-based normalization.
    #   +: Ignores more dark outliers.
    #   -: Uses more of the darkest pixels.
    "lower_percentile": 1.0,

    # Upper percentile used by percentile-based normalization.
    #   +: Preserves more bright structures.
    #   -: Compresses very bright outliers.
    "upper_percentile": 99.0,


    # Denoising algorithm applied after normalization.
    "denoise_method": "gaussian",

    # Gaussian smoothing standard deviations (pixels).
    #   +: More smoothing, less speckle, but reduced spatial detail.
    #   -: Preserves fine structures but retains more noise.
    "gaussian_sigma": (
        1.0,   # depth (z)
        0.5,   # horizontal (x)
    ),
}

In [ ]:
DETECTOR_CONFIG = {
    # Pixel-level structural feature extraction

    # Standard deviation of the Gaussian smoothing applied before computing local image gradients.
    #   +: Reduces speckle noise and produces smoother verticality maps but may blur fine barcode structures.
    #   -: Preserves fine detail but makes verticality estimates more sensitive to noise.
    "verticality_smoothing_sigma": 1.0,

    # Minimum verticality required for a pixel to be considered structurally oriented.
    #   +: Keeps only strongly vertical structures.
    #   -: Includes weaker or noisier vertical structures.
    "verticality_threshold": 0.60,

    # Gradient-magnitude quantile used to define structural pixels.
    #   +: Produces a smaller structural mask by retaining only the strongest gradients.
    #   -: Produces a larger structural mask by excluding more tissue.
    "gradient_quantile": 0.80,

    # Minimum connected-component size (pixels) retained in the structural mask.
    #   +: Removes isolated structural detections.
    #   -: Retains smaller structures.
    #  Zero disables size filtering.
    "minimum_component_size": 0,

    # Upper intensity quantile measured within each cleaned column.
    #   +: emphasize the brightest hypertransmission pixels.
    #   -: Smaller values measure a broader portion of the intensity distribution.
    "column_upper_quantile": 0.90,

    # Minimum number of remaining pixels required after structural exclusion for a column to be considered reliable.
    #   +: Rejects more columns.
    #   -: Allows noisier columns to contribute.
    "minimum_valid_pixels": 5,

    # Gaussian smoothing applied to column-level feature signals.
    #   +: Produces smoother detector responses and reduces isolated peaks.
    #   -: Preserves rapid local changes.
    "signal_smoothing_sigma": 2.0,

    # Multiplier applied to the IQR when thresholding column medians.
    #   +: More conservative threshold.
    #   -: More sensitive threshold.
    "median_iqr_multiplier": 1.0,

    # Multiplier applied to the IQR when thresholding the column upper-intensity statistic.
    #   +: Requires stronger hypertransmission.
    #   -: Detects weaker hypertransmission.
    "q90_iqr_multiplier": 0.5,

    # Horizontal window width (pixels) used when evaluating local depth continuity.
    #   +: Produces smoother continuity estimates over larger regions.
    #   -: Responds more quickly to local changes.
    "continuity_window_width": 15,

    # Maximum vertical displacement (pixels) considered when matching adjacent image columns.
    #   +: Allows continuity across larger vertical shifts.
    #   -: Requires tighter alignment between neighboring columns.  
    "continuity_depth_lag": 4,

    # Small numerical constant preventing division by zero in nearly onstant image regions.
    "continuity_minimum_row_standard_deviation": 1e-6,

    # Quantile threshold applied to the continuity score.
    #   +: Requires stronger depth continuity.
    #   -: Accepts weaker continuity.
    "continuity_quantile": 0.60,

    # Quantile threshold applied to the fraction of vertically organized pixels within candidate columns.
    #   +: Retains only highly vertical candidate columns.
    #   -: Allows less organized candidates.
    "vertical_fraction_quantile": 0.70,

    # Minimum horizontal interval length retained after thresholding.
    #   +: Removes short detections.
    #   -: Preserves smaller barcode candidates.
    "minimum_positive_run": 5,

    # Largest negative gap (pixels) filled between neighboring detections.
    #   +: Merges nearby intervals.
    #   -: Keeps intervals separate.
    #   Zero disables gap filling.
    "maximum_negative_gap": 2,

    # Number of image columns ignored at each lateral edge before interval extraction.
    #   +: Suppresses more edge artefacts.
    #   -: Includes more peripheral image content.
    "edge_margin": 10,
}

In [ ]:
calibration_cases = {}

for progression_group, subject_scans in SCAN_CONFIG.items():

    for subject_id, bscan_index in subject_scans.items():

        matches = [
            record
            for record in registry
            if (
                int(record.subject_id) == int(subject_id)
                and str(record.progression_group).lower()
                == progression_group.lower()
            )
        ]

        if not matches:
            raise KeyError(
                f"Could not find subject {subject_id} "
                f"in group '{progression_group}'."
            )

        if len(matches) > 1:
            raise ValueError(
                f"Subject {subject_id} has multiple matching volumes. "
                "Resolve the exact E2E file before calibration."
            )

        record = matches[0]

        volume = load_e2e_volume(
            record.e2e_path
        )

        if not 0 <= bscan_index < len(volume):
            raise IndexError(
                f"B-scan {bscan_index} is outside the valid range "
                f"0 to {len(volume) - 1} for subject {subject_id}."
            )

        processed = preprocess_bscan(
            volume=volume,
            bscan_index=bscan_index,
            **PREPROCESSING_CONFIG,
        )

        detector_result = (
            detect_structural_hypertransmission(
                processed.denoised_scan,
                config=DETECTOR_CONFIG,
            )
        )

        calibration_cases[
            (
                progression_group,
                subject_id,
            )
        ] = {
            "record": record,
            "bscan_index": int(
                bscan_index
            ),
            "processed": processed,
            "detector": detector_result,
        }

        print(
            f"Completed {progression_group} "
            f"subject {subject_id}, "
            f"B-scan {bscan_index}"
        )

## Preprocessed Scans

In [ ]:
for (
    progression_group,
    subject_id,
), case in calibration_cases.items():

    processed = case["processed"]

    image = processed.denoised_scan

    plt.figure(figsize=(14,5))

    plt.imshow(
        image,
        cmap="gray",
        aspect="auto",
    )

    plt.title(
        f"Subject {subject_id} | "
        f"{progression_group.title()} | "
        f"B-scan {case['bscan_index']}"
    )

    plt.xlabel("Horizontal position")
    plt.ylabel("Depth below BM")

    plt.tight_layout()

    plt.show()

## Automatic Detector

In [ ]:
for (
    progression_group,
    subject_id,
), case in calibration_cases.items():

    processed = case["processed"]

    detector = case["detector"]

    image = processed.denoised_scan

    h, w = image.shape

    plt.figure(figsize=(14,5))

    plt.imshow(
        image,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            w-0.5,
            h-0.5,
            -0.5,
        ),
    )

    color = (
        "tab:red"
        if progression_group == "fast"
        else "tab:green"
    )

    for interval in detector.intervals:

        plt.axvspan(
            interval.start,
            interval.end,
            color=color,
            alpha=0.30,
        )

    plt.title(
        f"Subject {subject_id} | "
        f"{progression_group.title()} | "
        f"B-scan {case['bscan_index']}"
    )

    plt.xlabel("Horizontal position")
    plt.ylabel("Depth below BM")

    plt.tight_layout()

    plt.show()

 create a separate notebook called something like manual_tuning.ipynb where you:

1. load the same 10 scans,
2. launch the manual annotator,
3. save the annotations,
4. compare manual vs. automatic intervals and begin tuning. This keeps the calibration dataset generation and the tuning process cleanly separated.